# Cell 1 — Setup

In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio

# -------------------------------------------------------------------
# INPUTS
# -------------------------------------------------------------------
points_gdb = r"E:\World Bank deliverbale 1\wb_buildings\buildings_points_utm.gdb"
points_layer = "wb_buildings_extent_point"

knn_folder = r"E:\World Bank deliverbale 1\knn"
summary_csv_folder = os.path.join(knn_folder, "summary_csvs")

# This should already exist from the previous workflow
knn_csv = os.path.join(summary_csv_folder, "knn_summary_all_k_wide.csv")

# -------------------------------------------------------------------
# OUTPUTS
# -------------------------------------------------------------------
out_gpkg = os.path.join(knn_folder, "buildings_points_knn_metrics.gpkg")
out_layer = "wb_buildings_extent_point_knn"

out_joined_csv = os.path.join(summary_csv_folder, "knn_metrics_for_join.csv")

print("Building points GDB:", points_gdb)
print("Building points layer:", points_layer)
print("KNN summary CSV:", knn_csv)
print("Output GeoPackage:", out_gpkg)

Building points GDB: E:\World Bank deliverbale 1\wb_buildings\buildings_points_utm.gdb
Building points layer: wb_buildings_extent_point
KNN summary CSV: E:\World Bank deliverbale 1\knn\summary_csvs\knn_summary_all_k_wide.csv
Output GeoPackage: E:\World Bank deliverbale 1\knn\buildings_points_knn_metrics.gpkg


# Cell 2 — Inspect the building point layer

In [2]:
print(pyogrio.list_layers(points_gdb))

[['wb_buildings_extent' 'MultiPolygon']
 ['wb_buildings_extent_point' 'Point']]


In [3]:
points = pyogrio.read_dataframe(points_gdb, layer=points_layer)

print("Rows:", f"{len(points):,}")
print("Columns:")
print(points.columns.tolist())
print(points.head())
print("CRS:", points.crs)

Rows: 578,379
Columns:
['type', 'building', 'construction', 'area_m_utm', 'ORIG_FID', 'geometry']
           type building construction  area_m_utm  ORIG_FID  \
0  multipolygon      yes         None  297.854874         1   
1  multipolygon      yes         None  134.675182         2   
2  multipolygon      yes         None  254.245943         3   
3  multipolygon      yes         None  331.438943         4   
4  multipolygon      yes         None  431.678814         5   

                        geometry  
0  POINT (344402.388 536236.527)  
1    POINT (343156.5 537298.568)  
2    POINT (343313.65 538071.93)  
3  POINT (346336.899 536569.205)  
4  POINT (343792.458 541426.962)  
CRS: EPSG:32636


# Cell 3 — Read the NN summary and create the ratio field

In [4]:
knn = pd.read_csv(knn_csv)

print("KNN rows:", f"{len(knn):,}")
print("KNN columns:")
print(knn.columns.tolist())
knn.head()

KNN rows: 578,379
KNN columns:
['building_id', 'neighbor_count_k5', 'mean_dist_k5', 'median_dist_k5', 'min_dist_k5', 'dist_to_5th_neighbor', 'std_dist_k5', 'max_near_rank_k5', 'complete_k5', 'neighbor_count_k10', 'mean_dist_k10', 'median_dist_k10', 'min_dist_k10', 'dist_to_10th_neighbor', 'std_dist_k10', 'max_near_rank_k10', 'complete_k10', 'neighbor_count_k20', 'mean_dist_k20', 'median_dist_k20', 'min_dist_k20', 'dist_to_20th_neighbor', 'std_dist_k20', 'max_near_rank_k20', 'complete_k20', 'neighbor_count_k40', 'mean_dist_k40', 'median_dist_k40', 'min_dist_k40', 'dist_to_40th_neighbor', 'std_dist_k40', 'max_near_rank_k40', 'complete_k40', 'neighbor_count_k80', 'mean_dist_k80', 'median_dist_k80', 'min_dist_k80', 'dist_to_80th_neighbor', 'std_dist_k80', 'max_near_rank_k80', 'complete_k80']


,building_id,neighbor_count_k5,mean_dist_k5,median_dist_k5,min_dist_k5,dist_to_5th_neighbor,std_dist_k5,max_near_rank_k5,complete_k5,neighbor_count_k10,...,max_near_rank_k40,complete_k40,neighbor_count_k80,mean_dist_k80,median_dist_k80,min_dist_k80,dist_to_80th_neighbor,std_dist_k80,max_near_rank_k80,complete_k80
0,1,5,21.221402,23.982673,15.893942,24.751932,4.283032,5,True,10,...,40,True,80,93.919224,104.300226,15.893942,139.705179,35.107359,80,True
1,2,5,10.927717,10.241977,9.041518,13.947397,1.858542,5,True,10,...,40,True,80,54.982228,63.170770,9.041518,85.227719,23.397487,80,True
2,3,5,13.970526,14.075989,11.625563,16.169194,2.207894,5,True,10,...,40,True,80,44.578165,48.961243,11.625563,64.679283,15.199920,80,True
3,4,5,21.542497,19.833185,9.761518,34.483712,9.266405,5,True,10,...,40,True,80,75.640316,80.254734,9.761518,113.930039,26.108108,80,True
4,5,5,26.281356,32.010138,0.933741,37.518905,14.634257,5,True,10,...,40,True,80,81.959188,79.085268,0.933741,139.469801,28.195560,80,True


# Now create the ratio field

In [5]:
# Make sure IDs are numeric
knn["building_id"] = pd.to_numeric(knn["building_id"], errors="coerce")

# Calculate ratio
knn["ratio_k80_to_k5"] = (
    knn["dist_to_80th_neighbor"] / knn["dist_to_5th_neighbor"]
)

# Replace infinite values with NaN, just in case
knn["ratio_k80_to_k5"] = knn["ratio_k80_to_k5"].replace([np.inf, -np.inf], np.nan)

# Keep only the fields you want to map
knn_keep = knn[
    [
        "building_id",
        "dist_to_5th_neighbor",
        "dist_to_80th_neighbor",
        "ratio_k80_to_k5",
    ]
].copy()

print(knn_keep.head())
print(knn_keep.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

   building_id  dist_to_5th_neighbor  dist_to_80th_neighbor  ratio_k80_to_k5
0            1             24.751932             139.705179         5.644213
1            2             13.947397              85.227719         6.110654
2            3             16.169194              64.679283         4.000155
3            4             34.483712             113.930039         3.303880
4            5             37.518905             139.469801         3.717321
         building_id  dist_to_5th_neighbor  dist_to_80th_neighbor  \
count  578379.000000         578379.000000          578379.000000   
mean   289190.000000             21.005756              92.707942   
std    166963.780009             14.568866              58.166055   
min         1.000000              2.529629              23.848991   
25%    144595.500000             13.634169              64.878347   
50%    289190.000000             17.504219              76.872505   
75%    433784.500000             23.767875             

# Cell 4 — Create quantile class fields

In [6]:
def add_quantile_class(df, value_col, q=4, out_col=None):
    """
    Adds quantile class field.
    Class 1 = lowest values.
    Class q = highest values.
    """
    if out_col is None:
        out_col = f"{value_col}_q{q}"

    valid = df[value_col].notna()

    df[out_col] = np.nan

    df.loc[valid, out_col] = (
        pd.qcut(
            df.loc[valid, value_col],
            q=q,
            labels=False,
            duplicates="drop"
        ) + 1
    )

    df[out_col] = df[out_col].astype("Int64")
    return df


for col in [
    "dist_to_5th_neighbor",
    "dist_to_80th_neighbor",
    "ratio_k80_to_k5",
]:
    knn_keep = add_quantile_class(knn_keep, col, q=4, out_col=f"{col}_q4")
    knn_keep = add_quantile_class(knn_keep, col, q=5, out_col=f"{col}_q5")

print(knn_keep.head())

   building_id  dist_to_5th_neighbor  dist_to_80th_neighbor  ratio_k80_to_k5  \
0            1             24.751932             139.705179         5.644213   
1            2             13.947397              85.227719         6.110654   
2            3             16.169194              64.679283         4.000155   
3            4             34.483712             113.930039         3.303880   
4            5             37.518905             139.469801         3.717321   

   dist_to_5th_neighbor_q4  dist_to_5th_neighbor_q5  dist_to_80th_neighbor_q4  \
0                        4                        4                         4   
1                        2                        2                         3   
2                        2                        3                         1   
3                        4                        5                         4   
4                        4                        5                         4   

   dist_to_80th_neighbor_q5  rat

# Export this joined-ready table as a CSV too:

In [7]:
knn_keep.to_csv(out_joined_csv, index=False)
print("Exported join table:", out_joined_csv)

Exported join table: E:\World Bank deliverbale 1\knn\summary_csvs\knn_metrics_for_join.csv


# Cell 5 — Join metrics to the building point layer

In [9]:
# -------------------------------------------------------------------
# Join metrics to the building point layer using ORIG_FID
# -------------------------------------------------------------------

join_field_points = "ORIG_FID"
join_field_knn = "building_id"

if join_field_points not in points.columns:
    raise ValueError(
        f"{join_field_points} was not found in the point layer columns. "
        f"Available columns are: {points.columns.tolist()}"
    )

if join_field_knn not in knn_keep.columns:
    raise ValueError(
        f"{join_field_knn} was not found in the KNN table columns. "
        f"Available columns are: {knn_keep.columns.tolist()}"
    )

# Make sure join keys are numeric integers
points[join_field_points] = pd.to_numeric(points[join_field_points], errors="coerce").astype("Int64")
knn_keep[join_field_knn] = pd.to_numeric(knn_keep[join_field_knn], errors="coerce").astype("Int64")

# Check uniqueness before joining
print("Point rows:", f"{len(points):,}")
print("Unique ORIG_FID values:", f"{points[join_field_points].nunique():,}")

print("KNN rows:", f"{len(knn_keep):,}")
print("Unique building_id values:", f"{knn_keep[join_field_knn].nunique():,}")

# Join
points_joined = points.merge(
    knn_keep,
    left_on=join_field_points,
    right_on=join_field_knn,
    how="left",
    validate="one_to_one"
)

print("\nOriginal point rows:", f"{len(points):,}")
print("Joined point rows:", f"{len(points_joined):,}")

# Check match rate
matched = points_joined["dist_to_5th_neighbor"].notna().sum()
unmatched = len(points_joined) - matched

print("Matched buildings:", f"{matched:,}")
print("Unmatched buildings:", f"{unmatched:,}")
print("Match rate:", f"{matched / len(points_joined):.2%}")

# Inspect any unmatched records
if unmatched > 0:
    print("\nExample unmatched records:")
    display(points_joined.loc[
        points_joined["dist_to_5th_neighbor"].isna(),
        [join_field_points, "geometry"]
    ].head())

Point rows: 578,379
Unique ORIG_FID values: 578,379
KNN rows: 578,379
Unique building_id values: 578,379

Original point rows: 578,379
Joined point rows: 578,379
Matched buildings: 578,379
Unmatched buildings: 0
Match rate: 100.00%


# Cell 6 — Clean field names for ArcGIS

In [10]:
rename_for_arcgis = {
    "dist_to_5th_neighbor": "d5_near",
    "dist_to_80th_neighbor": "d80_near",
    "ratio_k80_to_k5": "r80_5",
    "dist_to_5th_neighbor_q4": "d5_q4",
    "dist_to_5th_neighbor_q5": "d5_q5",
    "dist_to_80th_neighbor_q4": "d80_q4",
    "dist_to_80th_neighbor_q5": "d80_q5",
    "ratio_k80_to_k5_q4": "r80_5_q4",
    "ratio_k80_to_k5_q5": "r80_5_q5",
}

points_joined = points_joined.rename(columns=rename_for_arcgis)

print(points_joined.columns.tolist())

['type', 'building', 'construction', 'area_m_utm', 'ORIG_FID', 'geometry', 'building_id', 'd5_near', 'd80_near', 'r80_5', 'd5_q4', 'd5_q5', 'd80_q4', 'd80_q5', 'r80_5_q4', 'r80_5_q5']


# Cell 7 — Export joined points to GeoPackage

In [11]:
# Delete existing GPKG if you want a clean overwrite
if os.path.exists(out_gpkg):
    os.remove(out_gpkg)

points_joined.to_file(out_gpkg, layer=out_layer, driver="GPKG")

print("Exported joined point layer:")
print(out_gpkg)
print("Layer:", out_layer)

Exported joined point layer:
E:\World Bank deliverbale 1\knn\buildings_points_knn_metrics.gpkg
Layer: wb_buildings_extent_point_knn
